# Training MetricGAN+KAN
Mainly based on train.py from MetricGAN+.

In [ ]:
import os
import shutil
import sys
from enum import Enum, auto

import torch
import torchaudio
from hyperpyyaml import load_hyperpyyaml
from pesq import pesq

import speechbrain as sb
from speechbrain.dataio.sampler import ReproducibleWeightedRandomSampler
from speechbrain.nnet.loss.stoi_loss import stoi_loss
from speechbrain.processing.features import spectral_magnitude
from speechbrain.utils.distributed import run_on_main
from speechbrain.utils.metric_stats import MetricStats

# from MetricGAN_KAN import MetricDiscriminator, EnhancementGenerator

from pysepm import composite
from train import *

N_fft = 512
data_folder = "../data/noisy-vctk-16k"
output_folder = "../results"

# Requirements
```bash
pip install speechbrain
pip install https://github.com/schmiph2/pysepm/archive/master.zip
```

If `kaiser` is not found leading to an `ImportError`, you may need to go to line 2 of /path/to/your/python_env/site-packages/pysepm/utils.py, and change
```python
from scipy.signal import firls,kaiser,upfirdn
```
to
```python
from scipy.signal import firls,upfirdn
from scipy.signal.windows import kaiser
```

# Training

In [ ]:
def train_and_eval(model_param_file):
    hparams_file, run_opts, _ = sb.parse_arguments(model_param_file)
    model_name = model_param_file.split(".")[-2]
    with open(hparams_file) as fin:
        hparams = load_hyperpyyaml(fin, {"output_folder": os.path.join(output_folder, model_name), "data_folder": data_folder})

    # Initialize ddp (useful only for multi-GPU DDP training)
    sb.utils.distributed.ddp_init_group(run_opts)

    # Data preparation
    from voicebank_prepare import prepare_voicebank  # noqa

    run_on_main(
        prepare_voicebank,
        kwargs={
            "data_folder": hparams["data_folder"],
            "save_folder": hparams["data_folder"],
            "skip_prep": hparams["skip_prep"],
        },
    )

    # Create dataset objects
    datasets = dataio_prep(hparams)

    # Create experiment directory
    sb.create_experiment_directory(
        experiment_directory=hparams["output_folder"],
        hyperparams_to_save=hparams_file,
        overrides=overrides,
    )

    if hparams["use_tensorboard"]:
        from speechbrain.utils.train_logger import TensorboardLogger

        hparams["tensorboard_train_logger"] = TensorboardLogger(
            hparams["tensorboard_logs"]
        )

    # Create the folder to save enhanced files (+ support for DDP)
    run_on_main(create_folder, kwargs={"folder": hparams["enhanced_folder"]})

    se_brain = MGKBrain(
        modules=hparams["modules"],
        hparams=hparams,
        run_opts=run_opts,
        checkpointer=hparams["checkpointer"],
    )
    se_brain.train_set = datasets["train"]
    se_brain.historical_set = {}
    se_brain.noisy_scores = {}
    se_brain.batch_size = hparams["dataloader_options"]["batch_size"]
    se_brain.sub_stage = SubStage.GENERATOR

    if not os.path.isfile(hparams["historical_file"]):
        shutil.rmtree(hparams["MetricGAN_KAN_folder"])
    run_on_main(create_folder, kwargs={"folder": hparams["MetricGAN_KAN_folder"]})

    se_brain.load_history()
    # Load latest checkpoint to resume training
    se_brain.fit(
        epoch_counter=se_brain.hparams.epoch_counter,
        train_set=datasets["train"],
        valid_set=datasets["valid"],
        train_loader_kwargs=hparams["dataloader_options"],
        valid_loader_kwargs=hparams["valid_dataloader_options"],
    )

    # Load best checkpoint for evaluation
    test_stats = se_brain.evaluate(
        test_set=datasets["test"],
        max_key=hparams["target_metric"],
        test_loader_kwargs=hparams["dataloader_options"],
    )

    # Show parameters summary
    param_count = sum(p.numel() for p in se_brain.modules.generator.parameters() if p.requires_grad)
    se_brain.hparams.train_logger.log_stats({"Generator parameter count": param_count})
    param_count = sum(p.numel() for p in se_brain.modules.discriminator.parameters() if p.requires_grad)
    se_brain.hparams.train_logger.log_stats({"Discriminator parameter count": param_count})

In [ ]:
model_file_list = [f"hparams/train_{fname}" for fname in os.listdir("models") if fname.endswith('.yaml')]

for m in model_file_list:
    train_and_eval(m)
    break


In [ ]:
def generate_train_config():
    templates = None
    with open("hparams/train.yaml") as f:
        templates = f.readlines()
    for fname in os.listdir("models"):
        tmp = fname.split(".")
        if tmp[-1] == "yaml":
            with open(f"hparams/train_{tmp[-2]}.yaml", "w") as f:
                spec = templates.copy()
                spec[64] = f"models: !include:../models/{fname}\n"
                f.write("# Auto generated basing on train.yaml\n")
                f.writelines(spec)
            
# generate_train_config()